# Optimizing Agentic AI Collaboration v4.1.0 — L1 Majority Voting
## MATH-500 + MMLU, all 10 runs

**Hardware target:** none — CPU-only Kaggle session (no accelerator). This notebook does no fresh generation; every candidate answer is replayed from the 8 `sc-generations-*` artifacts the `v3.1.0`–`v3.3.1` notebooks already produced.

## What this notebook is

This notebook runs the **Majority Voting (MAJ)** protocol — L1's first heterogeneous multi-agent ensemble — over both **MATH-500** and **MMLU**, for all **5 ensembles** (`E1`–`E5`):
* `E1`: Qwen3-1.7B + Ministral-3-3B + Phi-4-mini
* `E2`: Qwen3-4B + Ministral-3-3B + Phi-4-mini
* `E3`: Qwen3-1.7B + Qwen3-4B + Phi-4-mini
* `E4`: all four models + a 2nd Qwen3-1.7B
* `E5`: all four models + a 2nd Qwen3-4B

That's **10 W&B runs total**, each with **2 reps** — an independent slot→generation draw per rep, giving a within-problem estimate of the ensemble's own sampling variance.

Candidates are **replayed**, never regenerated: `utils.ensembles.sample_slots` deterministically draws, per `(ensemble, dataset, problem_id, rep)`, one `gen_id ∈ range(5)` per slot from the already-downloaded SC artifacts, and `utils.graphs.build_majority_voting_graph` — the **same unmodified LangGraph template** the SC notebooks use in their own aggregation pass — groups and votes over the fanned-in candidates. `E2`'s results are directly comparable, rep-for-rep-0, to the `qwen3-1.7b`-verified BoN-MAV runs later in the campaign, since both draw the exact same candidate triples per problem.

## Relation with the project WPs

**WP2 - T4: L1 multi-agent protocol — Majority Voting.** The first consumer of the SC generation artifacts; `E2`'s candidate draws are also consumed by the BoN-MAV `v5.1.x` notebooks for a paired protocol-vs-protocol comparison. Latency/token/VRAM measurements (via `utils.voting.compose_generation_metrics`, the same function SC's own aggregation cell uses, in its heterogeneous-ensemble mode) feed the WP3 - T1 compute-accuracy trade-off analysis.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. **No accelerator needed** — leave Settings → Accelerator at **None**. This notebook is pure CPU (replay + grouping/voting only).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `WANDB_API_KEY` — Weights & Biases API key
3. **Before clicking Run All**, fill in the real pinned artifact versions in §3's `PRODUCER_REFS` — every `v3.1.0`–`v3.3.1` SC notebook must have already run and logged its `sc-generations-*` artifact.
4. Click **Run All**.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **No GPU needed** → Runtime → Change runtime type → CPU
2. **Add Colab Secrets** (🔑 icon in the left sidebar): `WANDB_API_KEY`, `GITHUB_TOKEN` (with "Notebook access" enabled for both)
3. Fill in `PRODUCER_REFS` in §3 before running, same as above.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `langgraph`, `datasets`, `wandb`, `math-verify`; `mistral-common` isn't needed here since no model is ever loaded). If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [1]:
# ── Environment detection: clone private repo ────────────────
# No CUDA_VISIBLE_DEVICES pinning here — unlike the SC/BoN-MAV notebooks, MAJ
# never loads a model, so there is no GPU to restrict.
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

ℹ️  Repo already present at /content/tfg-agentic-ai
repo_root = /content/tfg-agentic-ai


In [2]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

/content/tfg-agentic-ai


## 2. Authentication via Colab / Kaggle Secrets

We pull `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code. `HF_TOKEN` is not needed: this notebook never loads a model or tokenizer.

In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: colab
✅ Secrets loaded from Colab userdata.


## 3. Config

`MAJ_ENSEMBLES` (`E1`–`E5`) and the slot-sampling/naming helpers come from `utils.ensembles` — the single source of truth shared with the future BoN-MAV notebooks, so run names/artifact names/thesis tables cannot drift. `PRODUCER_REFS` pins all 8 SC generation artifacts **explicitly** (`entity/project/name:vN`, never `:latest`) — a MAJ run's lineage must never silently repoint at a newer/different SC artifact than the one it was actually built against.

In [4]:
import random

from utils.ensembles import (
    MAJ_ENSEMBLES, SLOT_SAMPLE_SEED, maj_matrices_artifact_name, maj_run_name,
    sample_slots, slot_agent_ids,
)
from utils.registry import MODELS, generations_artifact_name, get_model, problem_ref

PROJECT = "agentic-final-results" if (ENVIRONMENT in ("kaggle", "colab")) else "dev-l1"
print(f"W&B Project: {PROJECT}")

WANDB_ENTITY = "tfg-optimizing-agentic-ai"
SC_PROJECT = "agentic-final-results"

# Pinned producer artifact refs — explicit entity/project/name:vN, NEVER `:latest`.
# One per (model_key, dataset)
PRODUCER_REFS = {}
for model_key in MODELS:
    version = "v9" if "ministral" in model_key else "v4"
    for dataset in ("math500", "mmlu"):
        PRODUCER_REFS[(model_key, dataset)] = f"{WANDB_ENTITY}/{SC_PROJECT}/{generations_artifact_name(model_key, dataset)}:{version}"

version = "v9" if "ministral" in model_key else "v4"

TIE_BREAK_SEED = 2026
N_REPS = 2

print(f"MAJ_ENSEMBLES: {[ens.ensemble_id for ens in MAJ_ENSEMBLES]}")
print(f"N_REPS={N_REPS}, TIE_BREAK_SEED={TIE_BREAK_SEED}, SLOT_SAMPLE_SEED={SLOT_SAMPLE_SEED}")
print(f"\nPinned producer refs ({len(PRODUCER_REFS)}):")
for key, ref in PRODUCER_REFS.items():
    print(f"  {key[0]}/{key[1]}: {ref}")

W&B Project: agentic-final-results
MAJ_ENSEMBLES: ['E1', 'E2', 'E3', 'E4', 'E5']
N_REPS=2, TIE_BREAK_SEED=2026, SLOT_SAMPLE_SEED=3407

Pinned producer refs (8):
  qwen3-1.7b/math500: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-qwen3-1.7b-math500:v4
  qwen3-1.7b/mmlu: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-qwen3-1.7b-mmlu:v4
  qwen3-4b/math500: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-qwen3-4b-math500:v4
  qwen3-4b/mmlu: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-qwen3-4b-mmlu:v4
  ministral-3-3b/math500: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-ministral-3-3b-math500:v9
  ministral-3-3b/mmlu: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-ministral-3-3b-mmlu:v9
  phi-4-mini/math500: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-phi-4-mini-math500:v4
  phi-4-mini/mmlu: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-phi-4-mini-mmlu:v4

## 4. Data Loading

Both datasets are loaded once and shared across every ensemble/run. The evaluation subsets are drawn with `random.seed(789)` — identical to every SC/baseline notebook's own subset — so per-problem results pair exactly across protocols.

In [5]:
from utils.data import load_math500, load_mmlu
from utils.voting import MATH500_SPEC, MMLU_SPEC

DATA_SEED = 789
MATH_EVAL_SIZE = 100
MMLU_EVAL_SIZE = 128

math500_dataset = load_math500(split="test")
mmlu_dataset = load_mmlu(split="test")

random.seed(DATA_SEED)
math_test_sample = random.sample(list(math500_dataset), MATH_EVAL_SIZE)
random.seed(DATA_SEED)
mmlu_test_sample = random.sample(list(mmlu_dataset), MMLU_EVAL_SIZE)

# (dataset_key, dataset_sample, BenchmarkSpec, eval_size) — the one place both
# benchmarks' identities are bound; every downstream cell drives off this list.
RUN_DATASETS = [
    ("math500", math_test_sample, MATH500_SPEC, MATH_EVAL_SIZE),
    ("mmlu", mmlu_test_sample, MMLU_SPEC, MMLU_EVAL_SIZE),
]

print(f"✅ MATH-500: {len(math_test_sample)} problems sampled (seed={DATA_SEED})")
print(f"✅ MMLU: {len(mmlu_test_sample)} questions sampled (seed={DATA_SEED})")

✅ MATH-500: 100 problems sampled (seed=789)
✅ MMLU: 128 questions sampled (seed=789)


## 5. Reuse — Producer Model Keys

Every distinct `model_key` used by any `MAJ_ENSEMBLES` slot, across both benchmarks. The actual producer-artifact fetch (`utils.persistence.fetch_and_validate_stores`: `load_generations_artifact` → `build_generation_store` → `validate_alignment`, the same reuse pattern the `v3.0.0` smoke test's own round-trip stage proved works) now happens inside §6's per-run loop, once per real run via `run=run` — so every one of the 10 real MAJ runs records its own producer→consumer lineage, instead of a single upfront anonymous download shared across all of them.

In [6]:
from utils.persistence import fetch_and_validate_stores

# Every distinct model_key used by any MAJ_ENSEMBLES slot (not all 4 models are
# necessarily used by every ensemble, but the union across E1-E5 is all 4 here).
MAJ_MODEL_KEYS = sorted({mk for ens in MAJ_ENSEMBLES for mk in ens.slots})
print(f"Models used by MAJ_ENSEMBLES: {MAJ_MODEL_KEYS}")

Models used by MAJ_ENSEMBLES: ['ministral-3-3b', 'phi-4-mini', 'qwen3-1.7b', 'qwen3-4b']


## 6. Run Loop — 5 Ensembles × 2 Benchmarks = 10 MAJ Runs

One W&B run per `(ensemble, dataset)`. Each run: `fetch_and_validate_stores(distinct_model_keys, ..., run=run)` downloads + validates every distinct model artifact the ensemble uses — real producer→consumer lineage recorded on *this* run (`run.config` is then stamped with `alignment_validated`/`alignment_record_counts` via `run.config.update(...)`, since those aren't known until after the fetch) → one shared `maj_tie_rng = random.Random(TIE_BREAK_SEED)` for the whole run → per `(rep, problem_id)`: `sample_slots` → build `AgentSpec`s (`mode="replay"`) → `build_majority_voting_graph(specs, spec, rng=maj_tie_rng)` → `graph.invoke(...)` → score the vote's winner via `spec.score`/`spec.get_truth` (identical pattern to the SC generation-pass cell) → `compose_generation_metrics(out["samples"], tokens_per_s_rule="mean")` (**not** `out["metrics"]` — SC's own max-VRAM composition is wrong for a heterogeneous ensemble) → fill `(eval_size, N_REPS)` matrices.

**Per-rep recoverability is a hard requirement**, not an incidental side effect of the matrix shape: the results table's `rep` column makes every row individually filterable by problem *and* rep, and the npz artifact's matrices are indexable by rep column too, so any per-rep statistic (e.g. rep 0 alone, directly comparable to BoN-MAV's always-single-`rep=0` draw) is recoverable offline without re-running anything. Only the reps-combined statistics are pushed to `wandb.summary` — per-rep breakdowns are not duplicated there.

In [ ]:
import tempfile
import numpy as np
import wandb
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics, log_per_problem_means, log_stat_summary, finish_wandb
from utils.graphs import AgentSpec, build_majority_voting_graph
from utils.persistence import fetch_and_validate_stores
from utils.voting import compose_generation_metrics
from utils.statistics import (
    accuracy_statistics, time_statistics, token_statistics, vram_statistics, pct_votes_top_statistics,
    print_accuracy_stats_report, print_time_report, print_token_report, print_vram_report, print_pct_votes_top_report,
)

maj_run_summaries = []  # collected for the §7 cross-run summary

for dataset_key, dataset_sample, spec, eval_size in RUN_DATASETS:
    sample_prefix = "math_sample" if dataset_key == "math500" else "mmlu_sample" if dataset_key == "mmlu" else None

    for ens in MAJ_ENSEMBLES:
        N = len(ens.slots)
        agent_ids = slot_agent_ids(ens)
        distinct_model_keys = list(dict.fromkeys(ens.slots))
        print(f"\n{'='*60}\n  MAJ@{N} {ens.ensemble_id} — {dataset_key}\n{'='*60}")

        run = init_wandb(
            run_name=maj_run_name(ens.ensemble_id, dataset_key),
            project=PROJECT,
            config={
                "protocol": f"MAJ@{N}", "ensemble": ens.ensemble_id, "ensemble_slots": list(ens.slots),
                "slot_sample_seed": SLOT_SAMPLE_SEED, "tie_break_seed": TIE_BREAK_SEED, "n_reps": N_REPS,
                "producer_artifacts": {mk: PRODUCER_REFS[(mk, dataset_key)] for mk in distinct_model_keys},
                "eval_size": eval_size, "data_seed": DATA_SEED,
                "eval_dataset": dataset_key,
            },
            tags=["l1", "majority-voting", ens.ensemble_id, dataset_key],
        )
        # Own W&B run fetches its own copy of every producer artifact it depends
        # on, via load_generations_artifact(run=run) — real producer->consumer
        # lineage recorded on THIS run, not a bare use_artifact call plus reuse
        # of a shared upfront anonymous download. alignment_validated/_record_counts
        # aren't known until after the fetch, so they're stamped via
        # run.config.update(...) rather than the initial config= dict above.
        stores, alignment_counts = fetch_and_validate_stores(
            distinct_model_keys, lambda mk: PRODUCER_REFS[(mk, dataset_key)],
            run=run, dataset_sample=dataset_sample, render_task=spec.render_task,
            expected_dataset=dataset_key, expected_data_seed=DATA_SEED,
            expected_eval_size=eval_size, n_generators=5,
        )
        run.config.update({
            "alignment_validated": True,
            "alignment_record_counts": alignment_counts,
        })
        define_wandb_chart_only_metrics(prefixes=[f"{sample_prefix}/*"])

        maj_tie_rng = random.Random(TIE_BREAK_SEED)

        correctness_matrix = np.zeros((eval_size, N_REPS), dtype=int)
        time_matrix         = np.zeros((eval_size, N_REPS), dtype=float)
        input_tok_matrix    = np.zeros((eval_size, N_REPS), dtype=int)
        output_tok_matrix   = np.zeros((eval_size, N_REPS), dtype=int)
        total_tok_matrix    = np.zeros((eval_size, N_REPS), dtype=int)
        gpu_peak_matrix      = np.zeros((eval_size, N_REPS), dtype=float)
        tps_matrix           = np.zeros((eval_size, N_REPS), dtype=float)
        pct_votes_top_matrix = np.zeros((eval_size, N_REPS), dtype=float)

        results_columns = [
            "dataset", "ensemble", "protocol", "problem_id", "problem_ref", "rep", "slots",
            "task", "ground_truth", "final_output", "correct", "pct_votes_top", "tie", "n_abstain",
            "latency_s", "output_tokens", "total_tokens", "gpu_peak_mb", "throughput_tok_s",
        ]
        if dataset_key == "math500":
            results_columns += ["subject", "level"]
        results_rows = []
        n_tie = 0
        n_abstain_total = 0

        for rep in range(N_REPS):
            for problem_id, item in enumerate(dataset_sample):
                slots = sample_slots(ens, dataset=dataset_key, problem_id=problem_id, rep=rep)
                specs = [
                    AgentSpec(agent_id=aid, model_key=mk, model_id=get_model(mk)["model_id"], mode="replay",
                              store=stores[mk], gen_id=gid)
                    for aid, (mk, gid) in zip(agent_ids, slots)
                ]
                graph = build_majority_voting_graph(specs, spec, rng=maj_tie_rng)
                task = spec.render_task(item)
                ground_truth = spec.get_truth(item)
                out = graph.invoke({
                    "task": task, "problem_id": problem_id, "dataset": dataset_key,
                    "samples": [], "vote": None, "metrics": None,
                })
                vote = out["vote"]

                if vote["winner_idx"] is not None:
                    winner_sample = out["samples"][vote["winner_idx"]]
                    compare_value = (
                        winner_sample["answer"] if spec.compare_on == "answer" else winner_sample["parsed_output"]
                    )
                    correct = spec.score(compare_value, ground_truth)
                    winner_output = vote["winner_pred"]
                else:
                    correct = False
                    winner_output = None

                # compose_generation_metrics, NOT out["metrics"] — the graph's own
                # aggregate node still computes SC-style max-VRAM composition, which
                # under-counts a heterogeneous ensemble's true co-resident footprint.
                metrics = compose_generation_metrics(out["samples"], tokens_per_s_rule="mean")

                correctness_matrix[problem_id, rep] = int(correct)
                time_matrix[problem_id, rep]        = metrics["latency_s"]
                input_tok_matrix[problem_id, rep]   = metrics["input_tokens"]
                output_tok_matrix[problem_id, rep]  = metrics["output_tokens"]
                total_tok_matrix[problem_id, rep]   = metrics["total_tokens"]
                gpu_peak_matrix[problem_id, rep]    = metrics["gpu_peak_mb"]
                tps_matrix[problem_id, rep]         = metrics["tokens_per_s"]
                pct_votes_top_matrix[problem_id, rep] = vote["pct_votes_top"]

                if vote["tie"]:
                    n_tie += 1
                n_abstain_total += vote["n_abstain"]

                slots_str = ",".join(f"{mk}:{gid}" for mk, gid in slots)
                row = [
                    dataset_key, ens.ensemble_id, f"MAJ@{N}", problem_id, problem_ref(dataset_key, problem_id), rep,
                    slots_str, task, ground_truth, winner_output, correct,
                    round(vote["pct_votes_top"], 2), vote["tie"], vote["n_abstain"],
                    round(metrics["latency_s"], 2), metrics["output_tokens"], metrics["total_tokens"],
                    round(metrics["gpu_peak_mb"], 2), round(metrics["tokens_per_s"], 4),
                ]
                if dataset_key == "math500":
                    row += [item["subject"], item["level"]]
                results_rows.append(row)

        # One history point per problem, averaged over the reps, at
        # step == problem_id — the rep-major loop above makes two passes over
        # the same problems, and the per-sample charts must stay on the same
        # x-axis as the single-rep BoN-MAV runs. Per-rep values live in the
        # results table (`rep` column) and the npz artifact — not duplicated
        # into wandb.summary.
        log_per_problem_means(sample_prefix, {
            "accuracy":         correctness_matrix,
            "latency_s":        time_matrix,
            "output_tokens":    output_tok_matrix,
            "total_tokens":     total_tok_matrix,
            "gpu_peak_mb":      gpu_peak_matrix,
            "throughput_tok_s": tps_matrix,
            "pct_votes_top":    pct_votes_top_matrix,
        })

        results_table = wandb.Table(columns=results_columns, data=results_rows)
        wandb.log({f"results_{ens.ensemble_id}_{dataset_key.upper()}": results_table})

        # ── Statistics: combined (both reps) ────────────────────────────────────
        acc_stats   = accuracy_statistics(correctness_matrix)
        time_stats  = time_statistics(time_matrix)
        out_stats   = token_statistics(output_tok_matrix)
        total_stats = token_statistics(total_tok_matrix)
        vram_stats  = vram_statistics(gpu_peak_matrix)
        pct_votes_top_stats = pct_votes_top_statistics(pct_votes_top_matrix)

        print_accuracy_stats_report(f"MAJ@{N} {ens.ensemble_id} Accuracy ({dataset_key})", acc_stats)
        print_time_report(f"MAJ@{N} {ens.ensemble_id} Latency ({dataset_key})", time_stats)
        print_vram_report(f"MAJ@{N} {ens.ensemble_id} Peak VRAM ({dataset_key})", vram_stats)
        print_pct_votes_top_report(f"MAJ@{N} {ens.ensemble_id} Winner Vote Share ({dataset_key})", pct_votes_top_stats)

        n_total = eval_size * N_REPS
        wandb.summary["accuracy"]           = round(acc_stats["p_hat"], 6)
        wandb.summary["tie_rate"]           = round(n_tie / n_total, 4)
        wandb.summary["abstain_rate"]       = round(n_abstain_total / (n_total * N), 4)
        wandb.summary["mean_pct_votes_top"] = round(pct_votes_top_stats["mean_pct_votes_top"], 4)
        wandb.summary["mean_vram_peak"]     = round(vram_stats["mean_peak"], 0)
        wandb.summary["max_vram_peak"]      = round(vram_stats["peak_max"], 0)

        log_stat_summary("accuracy_stats", acc_stats,
            keys=["ci_lower", "ci_upper", "between_problem_std"])
        log_stat_summary("latency_stats", time_stats, keys=["ci_lower", "ci_upper", "between_problem_std", "flat_cv"])
        log_stat_summary("output_tokens_stats", out_stats, keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])
        log_stat_summary("total_tokens_stats", total_stats, keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])
        log_stat_summary("vram_peak_stats", vram_stats, keys=["peak_ci_lower", "peak_ci_upper", "between_problem_peak_std"])
        log_stat_summary("pct_votes_top_stats", pct_votes_top_stats, keys=["ci_lower", "ci_upper", "between_problem_std"])

        # ── Rep-0 accuracy — kept locally (not pushed to wandb.summary) for the
        # §7 cross-run comparison against BoN-MAV, which always draws a single
        # rep=0. Full per-rep breakdowns (accuracy/tie/abstain rate, vote share,
        # CIs) are recoverable from the results table's `rep` column and the
        # npz artifact's per-rep matrix columns, so they aren't duplicated here.
        rep0_acc_stats = accuracy_statistics(correctness_matrix[:, [0]])

        # ── Evaluation-matrices artifact (SC npz convention: maj{N}_-prefixed keys) ──
        artifact = wandb.Artifact(
            maj_matrices_artifact_name(ens.ensemble_id, dataset_key),
            type="evaluation-data",
            description=f"MAJ@{N} {ens.ensemble_id} evaluation matrices ({eval_size} x {N_REPS}). "
                        f"{dataset_key} with {ens.ensemble_id}.",
        )
        with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
            np.savez(
                f.name,
                **{
                    "problem_ids": np.arange(eval_size),
                    f"maj{N}_correctness":   correctness_matrix,
                    f"maj{N}_times":         time_matrix,
                    f"maj{N}_input_tokens":  input_tok_matrix,
                    f"maj{N}_output_tokens": output_tok_matrix,
                    f"maj{N}_total_tokens":  total_tok_matrix,
                    f"maj{N}_gpu_peak":      gpu_peak_matrix,
                    f"maj{N}_tps":           tps_matrix,
                },
            )
            artifact.add_file(f.name, name="evaluation_matrices.npz")
        wandb.log_artifact(artifact)

        maj_run_summaries.append({
            "ensemble": ens.ensemble_id, "dataset": dataset_key, "n_slots": N,
            "accuracy": acc_stats["p_hat"], "accuracy_rep0": rep0_acc_stats["p_hat"],
            "tie_rate": round(n_tie / n_total, 4), "mean_vram_peak": round(vram_stats["mean_peak"], 0),
            "run_url": run.url,
        })

        finish_wandb()
        print(f"✅ {ens.ensemble_id}/{dataset_key} MAJ run finished: accuracy={acc_stats['p_hat']:.3f}")

print(f"\n{'='*70}\n  ALL 10 MAJ RUNS COMPLETE (5 ensembles x 2 benchmarks)\n{'='*70}")

## 7. Cross-Run Summary & Analysis

A quick local table across all 10 runs — the authoritative reps-combined numbers live in W&B (each run's `accuracy_stats/*`); `acc_rep0` here is computed locally from the correctness matrix rather than read back from `wandb.summary`, since per-rep statistics aren't pushed there — the full per-rep breakdown lives in each run's results table (`rep` column) and its npz evaluation-matrices artifact. This is just a scan for anything obviously off before moving on.

In [8]:
print(f"\n{'dataset':<10} {'ensemble':<6} {'slots':<6} {'accuracy':<10} {'acc_rep0':<10} {'tie_rate':<10} {'mean_vram_mb':<14}")
print("-" * 70)
for s in maj_run_summaries:
    acc_rep0 = f"{s['accuracy_rep0']:.3f}" if s["accuracy_rep0"] is not None else "n/a"
    print(f"{s['dataset']:<10} {s['ensemble']:<6} {s['n_slots']:<6} {s['accuracy']:<10.3f} "
          f"{acc_rep0:<10} {s['tie_rate']:<10.3f} {s['mean_vram_peak']:<14.0f}")

print(f"\n✅ {len(maj_run_summaries)} MAJ runs logged. W&B run URLs:")
for s in maj_run_summaries:
    print(f"  {s['ensemble']}/{s['dataset']}: {s['run_url']}")

print(
    "\nNext steps:\n"
    "1. Record these 10 run URLs (plus the reuse round-trip run) for the thesis's MAJ results.\n"
    "2. E2's rep-0 candidate draws are directly comparable to the qwen3-1.7b-verified\n"
    "   BoN-MAV v5.1.x runs — a paired protocol-vs-protocol comparison, not two\n"
    "   independent draws.\n"
    "3. Proceed to the BoN-MAV v5.x notebooks (notebooks/bon-mav/), each preceded by\n"
    "   extending the v3.0.0 smoke test's Stage C calibration to ~10-20 real problems\n"
    "   before scheduling any real GPU session."
)


dataset    ensemble slots  accuracy   acc_rep0   tie_rate   mean_vram_mb  
----------------------------------------------------------------------
math500    E1     3      0.765      0.770      0.260      7046          
math500    E2     3      0.795      0.820      0.240      8408          
math500    E3     3      0.790      0.800      0.190      7331          
math500    E4     5      0.810      0.810      0.120      9895          
math500    E5     5      0.855      0.870      0.110      9897          
mmlu       E1     3      0.613      0.609      0.129      6926          
mmlu       E2     3      0.625      0.617      0.109      8276          
mmlu       E3     3      0.586      0.578      0.172      7257          
mmlu       E4     5      0.641      0.641      0.148      9732          
mmlu       E5     5      0.637      0.664      0.121      9738          

✅ 10 MAJ runs logged. W&B run URLs:
  E1/math500: https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/ew